loss is simply the difference between the actual value and predicted value, which is what we know from very long. now in real calculations you will have multiple losses to calculate, so in simple, you need to combine all of them to get one final loss, and that combining process is mostly done using some function naming __cost function__, also called as __loss function__ but its just another name there. not always you do some simply addition, but even though you try to add all of them you use the $\sum$ right, which becomes the basic cost function something like **MSE**: $\frac{1}{n}\sum(y - \hat{y})^2$ 

we do have multiple loss/cost functions based on which problem we are solving, lets see which are used mostly in DL:
#### regression:

In deep learning regression (e.g., predicting coordinates, housing prices, or continuous sensor signals), standard choices are:

* **Mean Squared Error (MSE / L2 Loss):**
    * **When it is used:** Default standard for most continuous targets.
    * **Why:** Has smooth, continuous derivatives that scale proportionally with error size, leading to fast convergence when data lacks extreme outliers.


* **Mean Absolute Error (MAE / L1 Loss / Smooth L1):**
    * **When it is used:** Financial forecasting, noisy sensor readings, or object detection bounding box regressions (where outliers would heavily distort an L2 loss).
    * **Why:** Robust to extreme outlier values.


* **Huber Loss / Smooth L1:**
    * **When it is used:** Modern computer vision models (e.g., Faster R-CNN, YOLO bounding box heads).
    * **Why:** Combines the fast quadratic convergence of MSE near zero with the linear outlier protection of MAE for large errors.

#### Classification 

In deep learning classification, models produce logits that pass through Sigmoid or Softmax:

* **Binary Cross-Entropy (BCE):**
    * **When it is used:** Two-class problems (Cat vs. Dog) OR multi-label classification (an image containing both a dog and a car simultaneously).
    * **Activation paired with:** **Sigmoid** on the output neuron(s).


* **Categorical Cross-Entropy (CCE) / Sparse Categorical Cross-Entropy:**
    * **When it is used:** Multi-class classification where each sample belongs to exactly one class (e.g., MNIST digit recognition, ImageNet).
    * **Activation paired with:** **Softmax** on the output layer.
    * *Note:* Use `SparseCategoricalCrossentropy` when labels are integers (`0, 1, 2, ...`), and standard `CategoricalCrossentropy` when labels are one-hot encoded vectors (`[1, 0, 0]`).


* **Focal Loss:**
    * **When it is used:** Extreme class imbalance (e.g., medical pathology detection, object detection background-versus-object imbalance).
    * **Why:** Down-weights the loss assigned to easy, well-classified examples so the model focuses gradient updates on rare, hard classes.


some of them you know might miss here, because these are mostly used, and your's might be rarely used.

now lets do some coding:


In [15]:
import numpy as np

y_predicted = np.array([1,1,0,0,1])
y_true = np.array([0.30,0.7,1,0,0.5])

In [16]:
def mae(y_true, y_predicted):
    total_error = 0
    for yt, yp in zip(y_true, y_predicted):
        total_error += abs(yt - yp)
    print("total_error:", total_error)

    mae = total_error/ len(y_true)
    print('mae:', mae)
    return mae

In [17]:
mae(y_true, y_predicted)

total_error: 2.5
mae: 0.5


np.float64(0.5)

if we wanted to do the same with this numpy, its going to be super easy

In [18]:
# to see the difference we do simply this
y_predicted - y_true

array([ 0.7,  0.3, -1. ,  0. ,  0.5])

In [19]:
# to se absolute difference
np.abs(y_predicted - y_true)

array([0.7, 0.3, 1. , 0. , 0.5])

In [20]:
# now mae is mean absolute difference so we take a mean of it
np.mean(np.abs(y_predicted - y_true))

np.float64(0.5)

In [21]:
# and to get total error we can simply do sum
np.sum(np.abs(y_predicted - y_true))

np.float64(2.5)

## now lets understand log loss and implement that:

### 1. The Core Formula of Log Loss (Binary Cross-Entropy)

For a single training sample with ground-truth label $y \in \{0, 1\}$ and predicted probability $\hat{y} \in (0, 1)$:

$$\text{Loss}(y, \hat{y}) = - \Big[ y \ln(\hat{y}) + (1 - y) \ln(1 - \hat{y}) \Big]$$

For an entire dataset of $N$ samples, you take the average:

$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \Big[ y_i \ln(\hat{y}_i) + (1 - y_i) \ln(1 - \hat{y}_i) \Big]$$

---

### 2. Dissecting the Formula: Why Each Component Exists

The formula looks like a two-headed monster, but it is actually a **mathematical switch** that activates only one term at a time depending on what $y$ is.

#### Case A: When the True Label is $y = 1$

Substitute $y = 1$ directly into the equation:

$$\text{Loss} = - \Big[ (1) \cdot \ln(\hat{y}) + (1 - 1) \cdot \ln(1 - \hat{y}) \Big]$$

$$\text{Loss} = -\ln(\hat{y})$$

The right-hand term completely zeroes out. The loss depends purely on $-\ln(\hat{y})$.

#### Case B: When the True Label is $y = 0$

Substitute $y = 0$ directly into the equation:

$$\text{Loss} = - \Big[ (0) \cdot \ln(\hat{y}) + (1 - 0) \cdot \ln(1 - \hat{y}) \Big]$$

$$\text{Loss} = -\ln(1 - \hat{y})$$

The left-hand term vanishes, leaving only $-\ln(1 - \hat{y})$.

---

### 3. Why the Logarithm ($\ln$) and Negative Sign ($-$) are Used

Look at how the natural logarithm $\ln(p)$ behaves for probabilities between $0.0$ and $1.0$:

* $\ln(1.0) = 0$
* $\ln(0.5) \approx -0.693$
* $\ln(0.1) \approx -2.302$
* $\ln(0.01) \approx -4.605$
* $\lim_{p \to 0} \ln(p) = -\infty$

Because probabilities are fractions between $0$ and $1$, their logarithms are **always negative numbers** (or zero).

1. **Why the negative sign ($-$) is required:**
Optimization algorithms like gradient descent are built to **minimize** a positive cost function toward zero. Placing a negative sign in front flips negative log values into positive loss penalties ($-\ln(0.1) = +2.302$).
2. **The Asymmetric Penalty (Why It Crushes Misclassifications):**
Suppose the true label is $y = 1$:
* If the model predicts $\hat{y} = 0.99$ (confident & correct):
$\text{Loss} = -\ln(0.99) \approx \mathbf{0.01}$ (almost no penalty).
* If the model predicts $\hat{y} = 0.50$ (uncertain):
$\text{Loss} = -\ln(0.50) \approx \mathbf{0.69}$ (moderate penalty).
* If the model predicts $\hat{y} = 0.01$ (confident & completely wrong):
$\text{Loss} = -\ln(0.01) \approx \mathbf{4.61}$.
* If $\hat{y} \to 0$:
$\text{Loss} \to \mathbf{+\infty}$.



If a model predicts the wrong class with high confidence, Log Loss imposes an exponentially exploding penalty. This forces the optimizer to prioritize correcting high-confidence mistakes during backpropagation.

---

### 4. Why Not Just Use MSE for Classification?

If Mean Squared Error works for regression, why can't we use $(y - \hat{y})^2$ for classification?

There are two primary mathematical reasons:

#### Reason 1: MSE Creates a "Non-Convex" Surface with Sigmoid

In classification, $\hat{y}$ is generated by a sigmoid function: $\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$.

If you plug that into MSE:


$$\text{Loss}_{\text{MSE}} = (y - \sigma(z))^2$$

The derivative with respect to weight $w$ requires the chain rule:


$$\frac{\partial \text{Loss}}{\partial w} = 2(y - \sigma(z)) \cdot \sigma'(z) \cdot x$$

Recall the derivative of sigmoid: $\sigma'(z) = \sigma(z)(1 - \sigma(z))$.

When the neuron's prediction is badly wrong (e.g., $z = -10$, so $\sigma(z) \approx 0$, but true $y = 1$):


$$\sigma'(z) \approx 0 \cdot (1 - 0) = \mathbf{0}$$

Plugging that back into the MSE gradient:


$$\frac{\partial \text{Loss}}{\partial w} = 2(1 - 0) \cdot (\mathbf{0}) \cdot x = \mathbf{0}$$

This is the **vanishing gradient problem on the output neuron**. Even though the model made a massive mistake, its gradient is near zero. The network stalls and cannot learn.

---

#### Reason 2: Log Loss Mathematically Cancels Out the Sigmoid Derivative!

When you use Log Loss with Sigmoid:

$$\text{Loss} = - \Big[ y \ln(\sigma(z)) + (1 - y) \ln(1 - \sigma(z)) \Big]$$

Let's compute the derivative with respect to the pre-activation input $z$:

$$\frac{\partial \text{Loss}}{\partial z} = -\left[ \frac{y}{\sigma(z)} \cdot \sigma'(z) - \frac{1-y}{1 - \sigma(z)} \cdot \sigma'(z) \right]$$

Substitute the identity $\sigma'(z) = \sigma(z)(1 - \sigma(z))$:

$$\frac{\partial \text{Loss}}{\partial z} = -\left[ \frac{y}{\sigma(z)} \cdot \sigma(z)(1 - \sigma(z)) - \frac{1-y}{1 - \sigma(z)} \cdot \sigma(z)(1 - \sigma(z)) \right]$$

Notice how the denominators cancel out cleanly:

$$\frac{\partial \text{Loss}}{\partial z} = -\Big[ y(1 - \sigma(z)) - (1 - y)\sigma(z) \Big]$$

$$\frac{\partial \text{Loss}}{\partial z} = -\Big[ y - y\sigma(z) - \sigma(z) + y\sigma(z) \Big]$$

$$\frac{\partial \text{Loss}}{\partial z} = \sigma(z) - y = \mathbf{\hat{y} - y}$$

The non-linear sigmoid derivative completely disappears from the gradient.

The resulting gradient is simply:

$$\frac{\partial \text{Loss}}{\partial w} = (\mathbf{\hat{y} - y}) \cdot x$$

* If the error is large ($\hat{y} = 0.01$ but $y = 1$), the gradient is $-0.99 \cdot x$ (a massive, linear push to update weights).
* If the error is small ($\hat{y} = 0.99$ and $y = 1$), the gradient is $-0.01 \cdot x$ (a tiny nudge).

There is no saturation, no vanishing gradients, and the loss surface is strictly convex, guaranteeing smooth gradient descent updates.

---

### 5. Probabilistic Origin: Maximum Likelihood Estimation (MLE)

Log Loss is not an arbitrary formula invented through trial and error. It is derived directly from probability theory.

In a binary task, each sample follows a **Bernoulli distribution**:

$$P(Y = y \mid x) = \hat{y}^y (1 - \hat{y})^{1 - y}$$

* If $y = 1 \implies P = \hat{y}^1 (1 - \hat{y})^0 = \hat{y}$
* If $y = 0 \implies P = \hat{y}^0 (1 - \hat{y})^1 = (1 - \hat{y})$

To find the optimal network weights across all independent samples, you maximize the joint probability (**Likelihood**):

$$\text{Likelihood} = \prod_{i=1}^N \hat{y}_i^{y_i} (1 - \hat{y}_i)^{1 - y_i}$$

Multiplying thousands of probabilities together causes **floating-point underflow** (numbers like $10^{-50}$ round to zero).

To fix this, mathematicians take the **natural log** of both sides, turning products into sums:

$$\ln(\text{Likelihood}) = \sum_{i=1}^N \Big[ y_i \ln(\hat{y}_i) + (1 - y_i) \ln(1 - \hat{y}_i) \Big]$$

Because machine learning optimizers are designed to minimize loss rather than maximize likelihood, we multiply by $-1$:

$$\text{Log Loss} = - \ln(\text{Likelihood})$$

Log Loss is literally the **Negative Log-Likelihood (NLL)** of the true data distribution under the model's predictions.

---

### The Clamping Safeguard

In pure mathematics, $\ln(0) = -\infty$. If your model outputs a probability of exactly $0.0$ or $1.0$, evaluating $\ln(\hat{y})$ or $\ln(1 - \hat{y})$ causes a division-by-zero math failure, producing `-inf` or `NaN` and crashing the training pipeline.

Clamping is not part of the theoretical loss formula—it is an **engineering boundary guard**:

* It forces any value $\le 0.0$ up to a tiny positive number ($\epsilon$).
* It forces any value $\ge 1.0$ down to just below one ($1 - \epsilon$).
* It keeps the inputs strictly inside the safe open interval $(0, 1)$ without altering the true meaning of the prediction.

---

### How It Directly Updates Gradients

When a classification neuron uses a Sigmoid activation ($\hat{y} = \sigma(z)$) alongside Log Loss, applying the chain rule causes the non-linear derivative terms to cancel out completely:

$$\frac{\partial \text{Loss}}{\partial w} = (\hat{y} - y) \cdot x$$

This error term $(\hat{y} - y)$ feeds directly into gradient descent:

* **Target $y = 1$, Prediction $\hat{y} = 0.98$ (Accurate):**
Error is $0.98 - 1 = -0.02$. The gradient is tiny, leaving the weights essentially untouched.
* **Target $y = 1$, Prediction $\hat{y} = 0.05$ (Severely Wrong):**
Error is $0.05 - 1 = -0.95$. The gradient is large, driving a substantial weight update to correct the error.
* **Target $y = 0$, Prediction $\hat{y} = 0.90$ (False Positive):**
Error is $0.90 - 0 = +0.90$. The gradient drives weights downward to suppress that neuron's activation.

Because the derivative simplifies directly to $(\hat{y} - y)$, gradient updates are smooth, linear, and completely avoid the vanishing gradient trap seen when pairing Sigmoid with MSE.

---

### What `1e-15` Actually Is

`1e-15` is scientific notation for:

$$1 \times 10^{-15} = \frac{1}{10^{15}} = 0.\underbrace{00000000000000}_{14 \text{ zeros}}1$$

* It provides a negligible numerical offset that prevents $\ln(0)$ crashes while preserving virtually exact loss values.
* In standard 64-bit floating-point computing (`float64`), memory precision is limited to about 15 to 17 decimal places. Using $10^{-15}$ sits right at the limit of hardware precision—any smaller (like `1e-17`), and 64-bit float math rounds $1 - \epsilon$ straight back to $1.0$, nullifying the clamp.

In [22]:
import numpy as np

np.log([0])

'''
see here, if you try log zero you get infinity but if you take any other value it can output something.
'''

C:\Users\DEVA\AppData\Local\Temp\ipykernel_13168\2306351954.py:3: RuntimeWarning: divide by zero encountered in log
  np.log([0])


'\nsee here, if you try log zero you get infinity but if you take any other value it can output something.\n'

In [23]:
# lets take a value which is very close to zero and see the log value of that
np.log([0.000000000000001])

array([-34.53877639])

In [24]:
np.log([1])

array([0.])

In [ ]:
''' 
in case of label=0 and its log giving infinity, we took some value which is close to zero but not zero. so for doing that lets take epsilon with 15 zeros after point. 
'''
epsilon = 1e-15

In [28]:
y_predicted

array([1, 1, 0, 0, 1])

In [38]:
y_true

array([0.3, 0.7, 1. , 0. , 0.5])

In [29]:
# no lets do one thing, lets replace 1 with a value close to 1 and zero with closer value to zero. so lets take a ne predicted list
y_predicted_new = [max(i, epsilon) for i in y_predicted] 
y_predicted_new

# see how the zero value got replaced with 1e-15

[np.int64(1), np.int64(1), 1e-15, 1e-15, np.int64(1)]

> now replacing 1. now you might think, why are we also replacing 1 with a value close to 1. well, the explanation is here.

### Why Must We Clamp 1 (and Not Just 0)?
Look closely at the second half of the Log Loss formula:$$\text{Loss} = - \Big[ y \ln(\hat{y}) + (1 - y) \ln(\mathbf{1 - \hat{y}}) \Big]$$

Now suppose the true label is $y = 0$, but the model outputs a confident prediction of $\hat{y} = 1.0$:
1. The first term cancels out: $0 \cdot \ln(1.0) = 0$.
2. Look inside the second term:$$\ln(1 - \hat{y}) = \ln(1 - 1.0) = \mathbf{\ln(0)}$$

Just like having $\hat{y} = 0$ breaks the first term ($\ln(0)$), having $\hat{y} = 1$ forces the second term to evaluate $\ln(1 - 1) = \ln(0) = -\infty$.

Clamping $\hat{y}$ to $1 - \epsilon$ ensures you evaluate $\ln(1 - (1 - \epsilon)) = \ln(\epsilon)$, keeping the calculation safe from a crash or NaN.

In [30]:
y_predicted_new = [min(i, 1-epsilon) for i in y_predicted_new]
y_predicted_new

[0.999999999999999, 0.999999999999999, 1e-15, 1e-15, 0.999999999999999]

In [32]:
y_predicted_new = np.array(y_predicted_new)
y_predicted_new

array([1.e+00, 1.e+00, 1.e-15, 1.e-15, 1.e+00])

In [33]:
np.log(y_predicted_new)

array([-9.99200722e-16, -9.99200722e-16, -3.45387764e+01, -3.45387764e+01,
       -9.99200722e-16])

In [ ]:
# if you did the log for y_predicted array, you would get warning because there are direct zeros in the list. see this. so you might now get the reason to do that y_predicted_new
np.log(y_predicted)

C:\Users\DEVA\AppData\Local\Temp\ipykernel_13168\3973244681.py:2: RuntimeWarning: divide by zero encountered in log
  np.log(y_predicted)


array([  0.,   0., -inf, -inf,   0.])

In [35]:
-np.mean(y_true*np.log(y_predicted_new)+(1-y_true)*np.log(1-y_predicted_new))

np.float64(17.2696280766844)

In [36]:
def log_loss(y_true, y_predicted):
    y_predicted_new = [max(i,epsilon) for i in y_predicted]
    y_predicted_new = [min(i,1-epsilon) for i in y_predicted_new]
    y_predicted_new = np.array(y_predicted_new)
    return -np.mean(y_true*np.log(y_predicted_new)+(1-y_true)*np.log(1-y_predicted_new))

In [37]:
log_loss(y_true, y_predicted)

np.float64(17.2696280766844)

instead of writing this 
```python
y_predicted_new = [max(i, epsilon) for i in y_predicted]
y_predicted_new = [min(i, 1 - epsilon) for i in y_predicted_new]
```

we can simply write this
```python
y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
```
- Lower Bound (epsilon): Any value $\le 0$ gets clamped up to epsilon ($10^{-15}$).
- Upper Bound (1 - epsilon): Any value $\ge 1$ gets clamped down to 1 - epsilon ($0.999999999999999$).